# Config

> Where the tunnel goes, what we assume about it, and where files live.


In [ ]:
#| default_exp config

In [ ]:
#| export
import os
from pathlib import Path

import matplotlib as mpl
import pyproj
from dotenv import load_dotenv
from shapely.geometry import LineString, Point

In [ ]:
from fastcore.test import test_close, test_eq

## Files

Downloads are cached in `data/raw/`, intermediate results go in `data/derived/`, and the blog's charts in `figures/`.


In [ ]:
#| export
ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd().parent
RAW, DERIVED, FIGURES = ROOT / "data" / "raw", ROOT / "data" / "derived", ROOT / "figures"
for _d in (RAW, DERIVED, FIGURES):
    _d.mkdir(parents=True, exist_ok=True)

Two inputs need an HM Land Registry account, so we can't download them for you. Say where they are in a `.env` file
at the project root (copy `.env.example`):

- `NPS_DIR`: the National Polygon Service shapefiles (`LR_POLY_FULL_*.shp`), i.e. title plans.
- `COD_DIR`: the CCOD and OCOD files of company owners (`CCOD_FULL_*.zip`, `OCOD_FULL_*.zip`). Defaults to `data/`.


In [ ]:
#| export
load_dotenv(ROOT / ".env")
NPS_DIR = Path(os.environ.get("NPS_DIR", ROOT / "data" / "nps"))
COD_DIR = Path(os.environ.get("COD_DIR", ROOT / "data"))

In [ ]:
test_eq((ROOT / "pyproject.toml").exists(), True)

## Coordinates

Points are given as (longitude, latitude), which is how maps label them. Distances and areas are measured on the
British National Grid (BNG), whose units are metres.


In [ ]:
#| export
BNG, WGS84 = "EPSG:27700", "EPSG:4326"
_to_bng = pyproj.Transformer.from_crs(WGS84, BNG, always_xy=True)


def to_bng(lonlat: tuple[float, float]) -> tuple[float, float]:
    "Convert (lon, lat) to BNG (x, y) metres."
    return _to_bng.transform(*lonlat)

Big Ben is at grid reference TQ 30268 79640:


In [ ]:
test_close(to_bng((-0.12463, 51.50073)), (530_268, 179_640), eps=10)

## The route

The tunnel is an immersed tube on the route TfL sketched in 2017: it crosses the river between the mouth of Nelson
Dock in Rotherhithe and the mouth of the West India Docks impound lock inlet, and ramps up inside each dock to the
street. People get on at the head of each ramp: Rotherhithe Street at Nelson Dock, and Westferry Road at the inlet.
One Canada Square stands in for "the jobs".

In [ ]:
#| export
SOUTH_PORTAL = (-0.033756, 51.504017)  # head of Nelson Dock, Rotherhithe Street
SOUTH_RIVER = (-0.032385, 51.504066)  # mouth of Nelson Dock
NORTH_RIVER = (-0.027002, 51.503256)  # mouth of the impound lock inlet
NORTH_PORTAL = (-0.025926, 51.503157)  # head of the inlet, Westferry Road
CANARY_WHARF = (-0.019570, 51.504970)  # One Canada Square


def tunnel_line() -> LineString:
    "The route in plan, ramp head to ramp head via the two dock mouths, in BNG."
    return LineString([to_bng(p) for p in (SOUTH_PORTAL, SOUTH_RIVER, NORTH_RIVER, NORTH_PORTAL)])


def river_crossing() -> LineString:
    "The immersed section under the river, dock mouth to dock mouth, in BNG."
    return LineString([to_bng(SOUTH_RIVER), to_bng(NORTH_RIVER)])

In [ ]:
test_close(river_crossing().length, 384, eps=5)
test_close(tunnel_line().length, 555, eps=5)

## The ramps

There are no lifts: people walk or ride down a ramp, through the tube and up the other side. How long the ramps are
depends on how deep the tube is, which we can work out from three published sources (all heights in metres above
Ordnance Datum Newlyn, ODN):

- **The quay.** The Environment Agency's flood defence records put the river walls at both ends at 5.2 m to 5.8 m.
  We take 5.5 m.
- **The riverbed.** The Port of London Authority's chart 319 (Lower Pool to Limehouse Reach) shows the channel at the
  crossing at most 6.6 m below chart datum, and its tide table puts chart datum 3.35 m below ODN at West India Pier.
- **The tube.** Its roof sits under 1.5 m of cover below the deepest point, with a 1 m thick roof and 3.5 m of
  headroom beneath that.

In [ ]:
#| export
QUAY_ODN = 5.5  # EA river wall crests at both ends
CHART_DATUM_ODN = -3.35  # PLA chart 319, West India Pier
DEEPEST_BELOW_CD = 6.6  # PLA chart 319, soundings along the crossing
COVER_M, ROOF_M, HEADROOM_M = 1.5, 1.0, 3.5


def walkway_odn() -> float:
    "Height of the walkway at the tunnel's lowest point (m ODN)."
    return CHART_DATUM_ODN - DEEPEST_BELOW_CD - COVER_M - ROOF_M - HEADROOM_M


DEPTH_M = QUAY_ODN - walkway_odn()  # quay to walkway

In [ ]:
test_close(walkway_odn(), -15.95, eps=0.01)
round(DEPTH_M, 1)

21.4

TfL says many UK ramps are built at 1 in 20, so each ramp is about 430 m long. That is longer than either dock, so
the ramps would have to double back on themselves.

In [ ]:
#| export
RAMP_GRADIENT = 1 / 20


def ramp_length_m() -> float:
    "Length of each ramp."
    return DEPTH_M / RAMP_GRADIENT


def tunnel_length_m() -> float:
    "Distance travelled from ramp head to ramp head: two ramps plus the river crossing."
    return river_crossing().length + 2 * ramp_length_m()

In [ ]:
test_close(ramp_length_m(), 429, eps=1)
test_close(tunnel_length_m(), 1243, eps=10)

## Travel assumptions


In [ ]:
#| export
WALK_KMH = 4.8
CYCLE_KMH = 15.0
LIFT_DELAY_MIN = 1.5  # waiting for and riding a lift, e.g. at the Greenwich foot tunnel

## Study area

Street networks and house sales cover Rotherhithe, Bermondsey, Deptford, the Isle of Dogs and Wapping. The
land readjustment area is a circle around the south portal.


In [ ]:
#| export
STUDY_BBOX = (-0.085, 51.470, 0.012, 51.522)  # (west, south, east, north)
CATCHMENT_M = 750


def catchment() -> Point:
    "The readjustment circle around the south portal, in BNG. Notebook 02 trims it to the Rotherhithe bank."
    return Point(to_bng(SOUTH_PORTAL)).buffer(CATCHMENT_M)

In [ ]:
test_close(catchment().area / 1e4, 3.1416 * 0.75**2 * 100, eps=1)  # ~177 ha

## Money

All money is in 2025 prices.


In [ ]:
#| export
PRICE_YEAR = 2025

## Charts

One palette and style for every figure in the blog.


In [ ]:
#| export
COLOURS = {
    "tunnel": "#e8673a", "southwark": "#2b78d6", "housing": "#ee6a37", "other": "#1db07f", "rest": "#b8b8b0", "private": "#dcdcd6",
    "water": "#d6e4ef", "land": "#f0efea", "park": "#cfe3c8", "jubilee": "#7b868c", "text": "#555555",
}  # fmt: skip
BACKGROUND = "#fbfbf9"


def use_style():
    "Apply the blog's chart style to matplotlib."
    mpl.rcParams.update({
        "figure.facecolor": BACKGROUND, "axes.facecolor": BACKGROUND, "savefig.facecolor": BACKGROUND,
        "savefig.dpi": 150, "savefig.bbox": "tight", "font.size": 11, "legend.frameon": False,
        "axes.titlesize": 15, "axes.titleweight": "bold", "axes.titlelocation": "left",
        "axes.spines.top": False, "axes.spines.right": False, "axes.edgecolor": "#aaaaaa",
        "axes.labelcolor": COLOURS["text"], "xtick.color": COLOURS["text"], "ytick.color": COLOURS["text"],
    })  # fmt: skip